<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/997_WDOv3_IntegrationTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Integration Tests

# 🧠 **What You Did Extremely Well**

## ✅ 1. **True End-to-End Graph Testing**

```python
graph.invoke(initial)
```

👉 This validates the full pipeline:

```text
config → graph → nodes → utilities → report → file
```

This is **real system validation**, not mocked behavior.

---

## ✅ 2. **Minimal Data Fixture (Perfect Design)**

```python
_write_minimal_data_dir(...)
```

👉 This is exactly right:

* small
* deterministic
* reusable
* fast

> This is how professional test suites are built.

---

## ✅ 3. **State Contract Validation (Critical)**

You check:

```python
result["workforce_metrics"]["status"] == "ok"
result["recommendations"]
isinstance(result["recommendations"][0], dict)
```

👉 This ensures:

* structure is preserved
* system outputs remain usable

---

## ✅ 4. **File Output Validation**

```python
assert Path(path).exists()
```

👉 You’re validating:

* side effects
* persistence
* actual output

This is **high-value testing**

---

## ✅ 5. **Error Resilience (Very Strong)**

### You tested:

* seed error preservation
* loader failure fallback

👉 This proves:

> The system does not break under failure

---

## ✅ 6. **Optional Real Data Smoke Test**

```python
@pytest.mark.skipif(...)
```

👉 This is **excellent practice**

* allows local validation
* avoids CI dependency

---

# 🏆 **Where You Are Now**

You have:

| Layer             | Status |
| ----------------- | ------ |
| Utilities         | ✅      |
| Nodes             | ✅      |
| Integration       | ✅      |
| Failure paths     | ✅      |
| Output validation | ✅      |

👉 This is **full-stack testing coverage**

---

# ⚠️ **Only 3 High-Value Additions Left**

These are small but meaningful.

---

# 🔥 1. **Config Propagation in Integration (CRITICAL)**

Right now you test config in utilities/nodes —
but not through the full graph.

---

## Add:

```python
def test_config_changes_affect_full_graph(tmp_path):
    data_dir = tmp_path / "data"
    data_dir.mkdir()
    (tmp_path / "reports_out").mkdir()

    _write_minimal_data_dir(data_dir)

    cfg = WDOv3OrchestratorConfig(
        readiness_delta_min_for_improving=100.0  # impossible threshold
    )
    cfg.data_dir = "data"
    cfg.reports_dir = "reports_out"
    cfg.project_root = str(tmp_path)

    graph = create_wdo_v3_orchestrator(cfg, project_root=tmp_path)
    initial = build_wdo_v3_initial_state({"project_root": str(tmp_path)})

    result = graph.invoke(initial)

    assert result["workforce_metrics"]["trajectory_status"] != "improving"
```

---

## 🎯 Why this matters:

👉 Validates your **core architecture end-to-end**

---

# 🔥 2. **Mixed Signals Integration Test**

You added this feature — test it at system level.

---

## Add:

```python
def test_mixed_signals_integration(tmp_path):
    data_dir = tmp_path / "data"
    data_dir.mkdir()
    (tmp_path / "reports_out").mkdir()

    # readiness ↑ but gaps ↑
    snapshots = [
        {"run_id": "R1", "date": "2026-01-01", "department_id": "D1",
         "readiness_score": 60, "skill_gap_index": 0.3,
         "avg_automation_exposure": 0.5, "reskilling_progress_pct": 0.2,
         "attrition_risk_index": 0.3},
        {"run_id": "R2", "date": "2026-02-01", "department_id": "D1",
         "readiness_score": 65, "skill_gap_index": 0.35,
         "avg_automation_exposure": 0.55, "reskilling_progress_pct": 0.25,
         "attrition_risk_index": 0.3},
    ]

    (data_dir / "workforce_snapshots.json").write_text(json.dumps(snapshots))
    (data_dir / "departments.json").write_text(json.dumps([{"department_id": "D1", "headcount": 10}]))

    for name in ["skills_gaps.json","automation_signals.json","training_investments.json",
                 "workforce_risk_controls.json","workforce_scenarios.json",
                 "workforce_snapshot_drivers.json","employees.json"]:
        (data_dir / name).write_text("[]")

    cfg = WDOv3OrchestratorConfig()
    cfg.data_dir = "data"
    cfg.reports_dir = "reports_out"
    cfg.project_root = str(tmp_path)

    graph = create_wdo_v3_orchestrator(cfg, project_root=tmp_path)
    result = graph.invoke(build_wdo_v3_initial_state({"project_root": str(tmp_path)}))

    assert result["workforce_metrics"]["trajectory_status"] == "mixed_signals"
```

---

## 🎯 Why this matters:

👉 Protects your **most nuanced feature**

---

# 🔥 3. **Report Quality Contract (Executive-Level Test)**

Right now you check:

```python
"Verdict:" in md
```

Upgrade this slightly.

---

## Add:

```python
assert "What this means" in md
assert "Key signals" in md
assert "Time sensitivity" in md
assert "Executive ask" in md
```

---

## 🎯 Why this matters:

👉 Ensures your **CEO-grade reporting standard never regresses**

---

# 🧠 **What You Should NOT Add**

Avoid:

* ❌ testing LangGraph internals
* ❌ over-testing file contents
* ❌ snapshot testing markdown

---

# 🏆 **Final Assessment**

## You are now at:

> 🟢 **Enterprise-grade system testing**

Not exaggeration — this is **real production-level coverage**





In [ ]:
"""WDO v3 integration tests: full LangGraph invoke, state contract, report file."""

from __future__ import annotations

import json
import sys
from pathlib import Path
from unittest.mock import patch

import pytest

root = Path(__file__).resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from config import WDOv3OrchestratorConfig

from agents.wdo_v3.orchestrator.orchestrator import (
    build_wdo_v3_initial_state,
    create_wdo_v3_orchestrator,
)


def _write_minimal_data_dir(data_dir: Path) -> None:
    """Minimal JSON set for one department, two periods (matches utilities fixture scale)."""
    snapshots = [
        {
            "run_id": "R1",
            "date": "2026-01-31",
            "department_id": "D1",
            "avg_automation_exposure": 0.5,
            "skill_gap_index": 0.4,
            "reskilling_progress_pct": 0.2,
            "attrition_risk_index": 0.3,
            "readiness_score": 60,
        },
        {
            "run_id": "R2",
            "date": "2026-02-28",
            "department_id": "D1",
            "avg_automation_exposure": 0.55,
            "skill_gap_index": 0.35,
            "reskilling_progress_pct": 0.35,
            "attrition_risk_index": 0.25,
            "readiness_score": 65,
        },
    ]
    (data_dir / "workforce_snapshots.json").write_text(json.dumps(snapshots), encoding="utf-8")
    (data_dir / "departments.json").write_text(
        json.dumps([{"department_id": "D1", "headcount": 10, "name": "Integration Dept"}]),
        encoding="utf-8",
    )
    for name, payload in (
        ("skills_gaps.json", []),
        ("automation_signals.json", []),
        ("training_investments.json", []),
        ("workforce_risk_controls.json", []),
        ("workforce_scenarios.json", []),
        ("workforce_snapshot_drivers.json", []),
        ("employees.json", []),
    ):
        (data_dir / name).write_text(json.dumps(payload), encoding="utf-8")


def test_wdo_v3_graph_invoke_happy_path(tmp_path):
    data_dir = tmp_path / "data"
    data_dir.mkdir()
    reports_rel = "reports_out"
    (tmp_path / reports_rel).mkdir()

    _write_minimal_data_dir(data_dir)

    cfg = WDOv3OrchestratorConfig()
    cfg.data_dir = "data"
    cfg.reports_dir = reports_rel
    cfg.project_root = str(tmp_path)

    graph = create_wdo_v3_orchestrator(cfg, project_root=tmp_path)
    initial = build_wdo_v3_initial_state({"project_root": str(tmp_path)})
    result = graph.invoke(initial)

    assert result.get("errors") == []
    path = result.get("report_file_path")
    assert path and Path(path).exists()
    md = Path(path).read_text(encoding="utf-8")
    assert "Workforce Development Orchestrator v3" in md
    assert "Verdict:" in md
    assert result.get("workforce_metrics", {}).get("status") == "ok"
    assert result.get("recommendations")
    assert isinstance(result["recommendations"][0], dict)
    assert "Executive ask:" in md

    g = graph.get_graph()
    for nid in ("goal", "planning", "data_loading", "workforce_analysis", "report"):
        assert nid in g.nodes


def test_wdo_v3_invoke_preserves_seed_error_and_still_writes_report(tmp_path):
    data_dir = tmp_path / "data"
    data_dir.mkdir()
    reports_rel = "reports_out"
    (tmp_path / reports_rel).mkdir()
    _write_minimal_data_dir(data_dir)

    cfg = WDOv3OrchestratorConfig()
    cfg.data_dir = "data"
    cfg.reports_dir = reports_rel
    cfg.project_root = str(tmp_path)

    graph = create_wdo_v3_orchestrator(cfg, project_root=tmp_path)
    initial = build_wdo_v3_initial_state({"project_root": str(tmp_path)})
    initial["errors"] = ["seed_warning"]

    result = graph.invoke(initial)
    assert "seed_warning" in (result.get("errors") or [])
    assert result.get("report_file_path")


@pytest.mark.skipif(
    not (root / "agents" / "data" / "workforce_snapshots.json").exists(),
    reason="agents/data smoke optional",
)
def test_wdo_v3_invoke_real_agents_data_smoke(tmp_path):
    """Optional smoke against repo agents/data when present (local / CI with data)."""
    reports_rel = "reports_smoke"
    (tmp_path / reports_rel).mkdir()

    cfg = WDOv3OrchestratorConfig()
    cfg.data_dir = "agents/data"
    cfg.reports_dir = reports_rel
    cfg.project_root = str(root)

    graph = create_wdo_v3_orchestrator(cfg, project_root=root)
    initial = build_wdo_v3_initial_state({"project_root": str(root)})
    result = graph.invoke(initial)

    assert result.get("errors") == []
    assert result.get("report_file_path")
    md = Path(result["report_file_path"]).read_text(encoding="utf-8")
    assert "Data traceability" in md


def test_wdo_v3_degraded_loader_still_returns(tmp_path):
    data_dir = tmp_path / "data"
    data_dir.mkdir()
    (tmp_path / "reports_out").mkdir()

    cfg = WDOv3OrchestratorConfig()
    cfg.data_dir = "data"
    cfg.reports_dir = "reports_out"
    cfg.project_root = str(tmp_path)

    graph = create_wdo_v3_orchestrator(cfg, project_root=tmp_path)
    initial = build_wdo_v3_initial_state({"project_root": str(tmp_path)})

    # Patch where the data-loading node holds the reference (imported into nodes).
    with patch(
        "agents.wdo_v3.orchestrator.nodes.load_wdo_v3_data_bundle",
        side_effect=OSError("boom"),
    ):
        result = graph.invoke(initial)

    assert any("Data loading failed" in e for e in (result.get("errors") or []))
